La última pieza para comprender el algoritmo de atención completo es dividir las matrices de atención en submatrices «altas y delgadas» que procesen los vectores de tokens por separado y en paralelo: lo que llamamos **Multi-Head Attention (MHA)**. Una vez comprendido el mecanismo de una sola cabeza, pasar a múltiples cabezas es un paso conceptual muy directo.

---

### 1. La proyección inicial y la mezcla dimensional

Como vimos antes, la matriz de consultas $Q$ es el producto de la matriz de entrada $X$ (los token embeddings de dimensión `[sequence_length, n_embd]`) por la matriz de pesos aprendible $W_Q$:

$$Q = X W_Q$$

Cada elemento en $Q$ es el resultado de productos punto que combinan la información de todas las dimensiones de los embeddings. Por lo tanto, no existe una correspondencia directa donde, por ejemplo, «las primeras 10 dimensiones de $X$ correspondan a las primeras 10 columnas de $Q$». En su lugar, la matriz de pesos $W_Q$ proyecta y redistribuye toda la información a lo largo de la matriz $Q$.

---

### 2. Trocear la matriz: El nacimiento de las cabezas

En una sola cabeza (*Single-Head*), la matriz $Q$ completa se multiplica por $K^T$ para calcular la atención. En **Multi-Head Attention**, dividimos (*split/slice*) estas matrices proyectadas a lo largo del eje de los canales antes de calcular la atención:

* Obtenemos particiones disjuntas (no solapadas): $Q_1, Q_2, \dots, Q_h$ (junto con sus respectivos $K_i$ y $V_i$).
* **Condición de divisibilidad:** Todas las cabezas deben tener exactamente el mismo tamaño. Por lo tanto, la dimensión del embedding ($d$) debe ser divisible de manera exacta entre el número de cabezas ($h$):

$$d_k = \frac{d}{h}$$



*(Por ejemplo: si $d = 100$, no podemos usar 3 cabezas; con 4 cabezas, cada una tendrá un ancho $d_k = 25$).*

---

### 3. La ecuación de atención por cabeza y la escala corregida

Cada cabeza ejecuta la ecuación de atención en su propio subespacio:

$$\text{head}_i = \text{Attention}(Q_i, K_i, V_i) = \text{softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}} + M\right) V_i$$

* **Máscara Causal ($M$):** Se mantiene idéntica en cada cabeza porque opera sobre las dimensiones cuadradas de la secuencia (`sequence_length × sequence_length`).
* **Factor de escala:** Ahora el divisor para controlar la varianza es $\sqrt{d_k} = \sqrt{d/h}$.

---

### 4. Concatenación y la matriz de proyección final ($W_O$)

La ecuación de cada cabeza no incluye $W_O$. Primero ejecutamos las $h$ cabezas en paralelo, luego **concatenamos horizontalmente** todas las matrices de activación resultantes y finalmente multiplicamos por la matriz de proyección de salida:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \text{head}_2, \dots, \text{head}_h) \, W_O$$

* **¿Para qué sirve $W_O$?** Cada cabeza se especializa en aprender relaciones distintas del texto (sintaxis, correferencia, dependencias a larga distancia). $W_O$ es la capa lineal encargada de comunicar y mezclar lo que cada cabeza aprendió de forma independiente, asegurando que la información fluya unificada hacia la siguiente subcapa.

---

### 5. De matrices 2D a tensores con Batches

En la implementación real en código, no trabajamos con simples matrices 2D, sino con **tensores multidimensionales** (por ejemplo: `[batch_size, num_heads, sequence_length, head_dim]`).

Esto requiere operaciones de *broadcasting* (difusión) para aplicar la máscara causal a través de todos los batches y cabezas simultáneamente. Aunque la manipulación de dimensiones añade un pequeño grado de complejidad en código, la matemática subyacente es exactamente la misma y permite aprovechar al máximo la paralelización en GPU.

In [1]:
# nota

### 1. La proyección lineal: ¿Qué significa realmente «rebanar» $Q$, $K$ y $V$?

Antes de hacer cualquier partición, los token embeddings ($X$) se multiplican por las matrices de proyección:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

Debido al producto matricial, **todas las dimensiones del espacio original de embeddings se mezclan por completo** en cada columna de $Q$, $K$ y $V$.

Por lo tanto:

* Cortar $Q$ en rebanadas (*slices*) de igual tamaño para formar las cabezas **no significa** que estemos tomando fragmentos aislados del vector original de tokens (por ejemplo, las dimensiones 1 a 64 del embedding).
* Cada rebanada representa un **subespacio proyectado y transformado** donde ya interactuó la totalidad del embedding inicial.
* Esto aplica de forma idéntica a $Q$, $K$ y $V$.

---

### 2. Multi-Head Attention no altera el conteo de parámetros

Un error conceptual común es pensar que tener múltiples cabezas añade más parámetros al modelo. **El número total de parámetros entrenables es exactamente el mismo que en Single-Head Attention.**

* **En Single-Head Attention:**
Se proyecta de $d_{\text{model}} \to d_{\text{model}}$.
$$\text{Parámetros de } W_Q = d_{\text{model}} \times d_{\text{model}}$$


* **En Multi-Head Attention (con $h$ cabezas de tamaño $d_k = d_{\text{model}} / h$):**
Cada cabeza proyecta a $d_k$. Si sumamos las $h$ cabezas:

$$h \times (d_{\text{model}} \times d_k) = h \times \left(d_{\text{model}} \times \frac{d_{\text{model}}}{h}\right) = d_{\text{model}} \times d_{\text{model}}$$



| Mecanismo | Parámetros Entrenables | Cómputo (FLOPs) |
| --- | --- | --- |
| **Single-Head Attention** | $4 \times d_{\text{model}}^2$ *(contando $W_Q, W_K, W_V, W_O$)* | Menor sobrecarga (un solo Softmax grande) |
| **Multi-Head Attention** | $4 \times d_{\text{model}}^2$ *(exactamente el mismo total)* | Mayor cómputo por operaciones paralelas y múltiples Softmax |

El modelo no gana capacidad por tener más pesos, sino por **diversificar la atención**: cada cabeza aprende a atender a diferentes tipos de relaciones semánticas o sintácticas en paralelo sin incrementar la memoria de parámetros.

---

### 3. El rol de la matriz de proyección final ($W_O$)

Una vez que cada cabeza $i$ procesa su atención de forma independiente:

$$\text{head}_i = \text{softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}} + M\right) V_i$$

Las activaciones resultantes se concatenan horizontalmente:

$$\text{Concat}(\text{head}_1, \text{head}_2, \dots, \text{head}_h)$$

La matriz $W_O$ (de dimensión $d_{\text{model}} \times d_{\text{model}}$) se encarga de realizar una **mezcla lineal** de todas estas salidas:

* **No altera destructivamente** lo que cada cabeza descubrió.
* **Sintetiza y redistribuye** la información recolectada por los distintos subespacios para entregar un vector unificado con la dimensión adecuada para el flujo residual y la siguiente subcapa.

Una cosa que quiero acotar es que diferentes interpretaciones del comportamiento de los LLMS no se basan en una teoria matematica formal. No se basan en restricciones que se imponen al modelo durante el entrenamiento. se usan las tecnicas de Interpretacion Mecanicista.

In [ ]:
# La capa infividual de leas Heads ayuda a la capa de MLP para identificar mas facil las caracteristicas y patrones

In [ ]:
# Aca no hay transformer block, MLP, Embeddings y datos - es solo para crear una tuberia de atencion multiple

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Hyperparametros

In [3]:
# Hiperparámetros de datos
seq_len = 8  # también conocido como ventana de contexto (context window)

# Hiperparámetros del modelo
embed_dim = 128
n_heads = 4  # embed_dim/n_heads debe ser un entero. Especificamos que el número de cabezales (heads) sea 4

# Hiperparámetros de entrenamiento
batch_size = 5

## Clase para atención multicabezal

In [4]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, embed_dim):
    super().__init__()

    # La dimensionalidad por cabezal es embed_dim dividida entre los cabezales
    self.num_heads = num_heads
    self.head_dim = embed_dim // num_heads

    # Matrices Q, K y V para num_heads, inicializadas como un "super-cabezal"
    #   nota: en el modelo 5, estas tres matrices se combinan en una sola (ejemplo *3)
    self.query = nn.Linear(embed_dim, embed_dim, bias=False)
    self.key = nn.Linear(embed_dim, embed_dim, bias=False)
    self.value = nn.Linear(embed_dim, embed_dim, bias=False)

    # La proyección lineal final fusiona las salidas de los cabezales
    self.W0 = nn.Linear(embed_dim, embed_dim, bias=False)

  def forward(self, x, track_sizes=False):

    # Extraer los tamaños de las dimensiones de entrada (embeddings de tokens)
    B, T, E = x.shape  # [batch, tokens (longitud de secuencia), embed_dim]
    if track_sizes:
      print(f"1){' Forma de los datos de entrada:':>32} {x.shape}")

    # Pasar datos a través de Q, K y V (múltiples cabezales aún en la misma matriz)
    q = self.query(x)  # [batch, seq_len, embed_dim]
    k = self.key(x)
    v = self.value(x)
    if track_sizes:
      print(f"2){'Forma de q/k/v antes de separar:':>32} {q.shape}")

    # Redimensionar para separar los cabezales (la separación se hace tras XW_Q)
    q = q.view(B, T, self.num_heads, self.head_dim)
    k = k.view(B, T, self.num_heads, self.head_dim)
    v = v.view(B, T, self.num_heads, self.head_dim)

    # La función SDPA de PyTorch requiere la forma [B, num_heads, T, head_dim]
    q = q.transpose(1, 2)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)
    if track_sizes:
      print(f"3){'Forma de q/k/v tras separar:':>32} {q.shape}")

    # Ahora podemos llamar a SDPA
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    if track_sizes:
      print(f"4){'Forma de datos post-atención:':>32} {out.shape}")

    # Nuestro código necesita regresar a [B, T, num_heads, head_dim]
    out = out.transpose(1, 2)
    if track_sizes:
      print(f"5){'Redimensión de datos post-atención:':>32} {out.shape}")

    # Fusionar los cabezales de vuelta en embed_dim
    out = out.reshape(B, T, E)
    if track_sizes:
      print(f"6){'Datos fusionados a tamaño:':>32} {out.shape}")

    # Finalmente, aplicar la matriz de mezcla lineal (W0)
    out = self.W0(out)
    if track_sizes:
      print(f"7){'Mezcla lineal W0 post-MHA:':>32} {out.shape}")

    return out

In [5]:
mha = MultiHeadAttention(n_heads,embed_dim)
mha

MultiHeadAttention(
  (query): Linear(in_features=128, out_features=128, bias=False)
  (key): Linear(in_features=128, out_features=128, bias=False)
  (value): Linear(in_features=128, out_features=128, bias=False)
  (W0): Linear(in_features=128, out_features=128, bias=False)
)

In [7]:
# Pasar algunos datos simulados
data = torch.randn(size=(batch_size, seq_len, embed_dim))
out = mha(data)
print(f'Tamaño de entrada:  {data.shape}')
print(f'Tamaño de salida:   {out.shape}')

Tamaño de entrada:  torch.Size([5, 8, 128])
Tamaño de salida:   torch.Size([5, 8, 128])


In [8]:
print(f' Longitud de secuencia: {seq_len:2d}')
print(f'Dimensión de embedding: {embed_dim}')
print(f'   Número de cabezales: {n_heads:2d}')
print(f' Dimensión por cabezal: {embed_dim // n_heads}')

print(
    '\nDimensiones de los datos a medida que pasan por la subcapa de atención'
    ' de un bloque Transformer:'
)
out = mha(data, track_sizes=True)

 Longitud de secuencia:  8
Dimensión de embedding: 128
   Número de cabezales:  4
 Dimensión por cabezal: 32

Dimensiones de los datos a medida que pasan por la subcapa de atención de un bloque Transformer:
1)  Forma de los datos de entrada: torch.Size([5, 8, 128])
2)Forma de q/k/v antes de separar: torch.Size([5, 8, 128])
3)    Forma de q/k/v tras separar: torch.Size([5, 4, 8, 32])
4)   Forma de datos post-atención: torch.Size([5, 4, 8, 32])
5)Redimensión de datos post-atención: torch.Size([5, 8, 4, 32])
6)      Datos fusionados a tamaño: torch.Size([5, 8, 128])
7)      Mezcla lineal W0 post-MHA: torch.Size([5, 8, 128])


**¿Cuántas Attention Heads se usan en modelos reales y de qué depende?**

La cantidad de cabezas de atención no se elige al azar; responde a cuatro factores matemáticos y de ingeniería de hardware:

* **La dimensión estándar por cabeza ($d_k$):** En la práctica, casi todos los modelos fijan la dimensión de cada cabeza en **$d_k = 64$** (modelos clásicos como BERT y GPT-2) o **$d_k = 128$** (el estándar en LLaMA, Mistral y Qwen). Por ende, el número de cabezas $h$ queda determinado automáticamente por la dimensión del embedding:

$$h = \frac{d_{\text{model}}}{d_k}$$


* **Paralelismo en GPUs (*Tensor Parallelism*):** Para distribuir un modelo en un clúster (por ejemplo, nodos de 8 GPUs H100/A100), el número de cabezas debe ser divisible uniformemente entre la cantidad de GPUs para partir las multiplicaciones matriciales sin cuellos de botella de sincronización. Por eso siempre vemos múltiplos de 8, 16, 32 o 64.
* **Mecanismo GQA (Grouped-Query Attention):** En arquitecturas modernas se desacoplan las cabezas: se mantienen muchas cabezas para los *Queries* (ej. 32 o 64) para preservar la capacidad expresiva, pero se agrupan en pocas cabezas para *Keys/Values* (típicamente 8) para reducir el consumo de memoria en el KV-Cache.
* **Leyes de escalado (*Scaling Laws*):** A medida que aumentan los parámetros y la dimensión del modelo ($d_{\text{model}}$ crece de 768 a 8192 o más), se incrementa el número de cabezas en lugar de hacer una sola cabeza gigantesca, evitando problemas de saturación en el producto punto.

---

### Tabla Comparativa: De la Era Clásica a los LLMs Actuales

| Modelo | Capas (*Layers*) | Attention Heads ($Q$) | KV Heads ($K,V$) | Dimensión ($d_{\text{model}}$) | Dim. Cabeza ($d_k$) | Parámetros |
| --- | --- | --- | --- | --- | --- | --- |
| **BERT-Base** | 12 | 12 | 12 (MHA) | 768 | 64 | 110M |
| **BERT-Large** | 24 | 16 | 16 (MHA) | 1024 | 64 | 340M |
| **GPT-2 (Small)** | 12 | 12 | 12 (MHA) | 768 | 64 | 124M |
| **GPT-2 (XL / 1.5B)** | 48 | 25 | 25 (MHA) | 1600 | 64 | 1.5B |
| **GPT-3** | 96 | 96 | 96 (MHA) | 12,288 | 128 | 175B |
| **LLaMA-2 (7B)** | 32 | 32 | 32 (MHA) | 4096 | 128 | 7B |
| **LLaMA-3 (8B)** | 32 | 32 | **8 (GQA)** | 4096 | 128 | 8B |
| **Mistral (7B)** | 32 | 32 | **8 (GQA)** | 4096 | 128 | 7B |
| **Mixtral 8x7B (MoE)** | 32 | 32 | **8 (GQA)** | 4096 | 128 | 46.7B |
| **Qwen 2.5 (7B)** | 28 | 28 | **4 (GQA)** | 3584 | 128 | 7.6B |
| **Gemma-2 (9B)** | 42 | 16 | **8 (GQA)** | 3584 | 256 | 9.2B |
| **LLaMA-3 (70B)** | 80 | 64 | **8 (GQA)** | 8192 | 128 | 70B |
| **Qwen 2.5 (72B)** | 80 | 64 | **8 (GQA)** | 8192 | 128 | 72.7B |

---

### Puntos Clave para la Publicación

* **La transición de MHA a GQA:** En los modelos clásicos (BERT, GPT-2, GPT-3), las cabezas de atención eran simétricas ($Q = K = V$). En los modelos modernos (LLaMA-3, Mistral, Qwen 2.5), las cabezas de $K$ y $V$ se reducen a 8 o 4 grupos para optimizar la inferencia sin perder precisión.
* **Estandarización de $d_k = 128$:** Mientras que la primera generación de Transformers operaba con cabezas de dimensión 64, prácticamente todos los modelos de escala moderna estandarizaron subespacios de 128 dimensiones por cabeza (o 256 en casos como Gemma).